# Notebook 08 — FFN: per-token nonlinearity

Paired with `theory/08_feedforward.md`. **Mode:** theory-first.

## QHMPC

**Question.** A sharp test of *what attention can't do that FFN can*: per-token nonlinear functions of the embedding. The canonical instance is XOR. Attention is a convex combination of linearly-projected values, so its output is **linear** in each token's embedding. XOR is not linearly separable. An attention-only block — even at full $d_{\text{model}}$ width — *cannot* compute per-token XOR; an FFN trivially can. We test this directly, plus the parameter accounting from §8.3, plus a width sweep on a memorization task.

**Hypothesis.** Attention-only model → XOR accuracy stuck at 50% (chance, since linear-in-embedding can't separate the four input patterns). FFN-only model (or attention+FFN) → solves to ~100%. This is the chapter's load-bearing claim made empirical.

**Method.**
1. Parameter / FLOP accounting (vanilla vs. SwiGLU at matched FLOPs).
2. **XOR experiment** — attention-only vs. FFN-only vs. both, on a per-token XOR task.
3. FFN-as-memory width sweep on a memorization task (where capacity, not nonlinearity, is the bottleneck).
4. Activation ablation (GELU / ReLU / SiLU).

**Prediction.** *Paste §8.7 predictions.*

**Confidence.** *[low / medium / high, reason]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Experiment 1 — parameter & FLOP accounting (Prop 8.4)

In [ ]:
d_model = 768
for ff_mult in [1, 2, 4, 8]:
    d_ff = ff_mult * d_model
    print(f'd_ff = {ff_mult}*d_model = {d_ff}:')
    print(f'  vanilla:  params = {2*d_model*d_ff:>10}   flops = {4*d_model*d_ff:>10}')
    print(f'  SwiGLU:   params = {3*d_model*d_ff:>10}   flops = {6*d_model*d_ff:>10}')
print(f'\nSwiGLU at matched FLOPs to vanilla(4x): d_ff = 8/3 * d_model ≈ {(8/3)*d_model:.0f}')

**Sub-finding 1.** *SwiGLU at matched FLOPs uses ~2.67× width (vs vanilla's 4×). The literature uses ~2.67× for SwiGLU.*

---
## Experiment 2 — Per-token XOR: attention can't, FFN can

**Task.** Vocabulary of 4 "input" tokens, each encoding one of the four 2-bit patterns $(0,0), (0,1), (1,0), (1,1)$. Sequence length 8 — at each position the model sees an input token and must predict the XOR of its two bits.

**Why this isolates per-token nonlinearity.** Attention's output at position $t$ is $\sum_j \alpha_{tj} W_V x_j$ — a linear function of each $x_j$. Stacking attention layers without FFNs keeps the output linear in the embedding (composition of linear maps). XOR of two binary features is not linearly separable (the canonical 1969 result). So an attention-only stack should provably plateau at 50% on this task; an attention+FFN model should solve it.

**Three model variants:** attention-only (FFN replaced by identity), FFN-only (attention replaced by identity), both. Same $d_{\text{model}}, d_{\text{ff}}$ in all cases.

All four input tokens are equally common; output is balanced 50/50 between XOR=0 and XOR=1.

In [ ]:
# 4 input tokens encoding the four 2-bit patterns.
PATTERNS = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
XORS = torch.tensor([0, 1, 1, 0])  # XOR of the two bits
V_IN = 4
SEQ_LEN = 8
D_MODEL = 16
D_FF = 32

def make_batch(B):
    seq = torch.randint(0, V_IN, (B, SEQ_LEN))
    targets = XORS[seq]                                      # (B, SEQ_LEN)
    return seq, targets

class Block(nn.Module):
    def __init__(self, attn_on=True, ffn_on=True):
        super().__init__()
        self.attn_on, self.ffn_on = attn_on, ffn_on
        self.ln1 = nn.LayerNorm(D_MODEL); self.ln2 = nn.LayerNorm(D_MODEL)
        if attn_on:
            self.attn = nn.MultiheadAttention(D_MODEL, num_heads=4, batch_first=True, bias=False)
        if ffn_on:
            self.ffn = nn.Sequential(nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL))
    def forward(self, x):
        if self.attn_on:
            a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False)
            x = x + a
        if self.ffn_on:
            x = x + self.ffn(self.ln2(x))
        return x

class XORModel(nn.Module):
    def __init__(self, attn_on=True, ffn_on=True):
        super().__init__()
        self.embed = nn.Embedding(V_IN, D_MODEL)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN, D_MODEL) * 0.02)
        self.block = Block(attn_on, ffn_on)
        self.out = nn.Linear(D_MODEL, 2, bias=False)
    def forward(self, seq):
        x = self.embed(seq) + self.pos
        return self.out(self.block(x))

def train_eval(attn_on, ffn_on, steps=2000, lr=3e-3):
    torch.manual_seed(0)
    m = XORModel(attn_on, ffn_on)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        seq, tgt = make_batch(64)
        loss = F.cross_entropy(m(seq).reshape(-1, 2), tgt.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        seq, tgt = make_batch(2000)
        acc = (m(seq).argmax(-1) == tgt).float().mean().item()
    return losses, acc

configs = [('attention only', True, False),
           ('FFN only',       False, True),
           ('both',           True, True)]
results = {}
for name, attn, ffn in configs:
    losses, acc = train_eval(attn, ffn)
    results[name] = (losses, acc)
    print(f'{name:>15}: final acc = {acc:.3f}   (chance = 0.5)')

fig, ax = plt.subplots(figsize=(7, 4))
for name, (losses, _) in results.items():
    ax.plot(losses, label=name)
ax.axhline(np.log(2), color='k', linestyle=':', alpha=0.5, label='chance loss = log 2')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log')
ax.set_title('XOR — attention can\'t, FFN can')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 2.** *Did attention-only plateau at chance loss $\log 2$? Did FFN-only solve cleanly? This is the empirical version of the linear-vs-nonlinear separation claim. If attention-only DOES solve it, something in the LayerNorm or residual is sneaking in nonlinearity — investigate.*

---
## Experiment 3 — FFN as memory: width sweep on a recall task

Different question now: holding nonlinearity fixed (FFN present), how does *width* gate memorization capacity? The FFN-as-key-value-memory view (Prop 8.5) predicts that performance climbs with $d_{\text{ff}}$ until there are enough memory slots for the task.

In [ ]:
V_KEY, V_VAL = 32, 32
N_PAIRS, SEQ_LEN_R = 5, 11
D_MODEL_R = 32
V_TOTAL_R = V_KEY + V_VAL

def make_recall_batch(B):
    keys = torch.randint(0, V_KEY, (B, N_PAIRS))
    vals = torch.randint(0, V_VAL, (B, N_PAIRS))
    qi = torch.randint(0, N_PAIRS, (B,))
    targets = vals[torch.arange(B), qi]
    seq = torch.zeros(B, SEQ_LEN_R, dtype=torch.long)
    for b in range(B):
        for p in range(N_PAIRS):
            seq[b, 2*p] = keys[b, p]
            seq[b, 2*p + 1] = V_KEY + vals[b, p]
        seq[b, -1] = keys[b, qi[b]]
    return seq, targets

class Recall(nn.Module):
    def __init__(self, d_ff, activation='gelu'):
        super().__init__()
        self.embed = nn.Embedding(V_TOTAL_R, D_MODEL_R)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN_R, D_MODEL_R) * 0.02)
        self.ln1 = nn.LayerNorm(D_MODEL_R); self.ln2 = nn.LayerNorm(D_MODEL_R)
        self.attn = nn.MultiheadAttention(D_MODEL_R, num_heads=4, batch_first=True, bias=False)
        act = {'gelu': nn.GELU(), 'relu': nn.ReLU(), 'silu': nn.SiLU()}[activation]
        self.ffn = nn.Sequential(nn.Linear(D_MODEL_R, d_ff), act, nn.Linear(d_ff, D_MODEL_R))
        self.out = nn.Linear(D_MODEL_R, V_VAL, bias=False)
    def forward(self, seq):
        x = self.embed(seq) + self.pos
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return self.out(x[:, -1])

def train_recall(d_ff, activation='gelu', steps=1500):
    torch.manual_seed(0)
    m = Recall(d_ff, activation)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for _ in range(steps):
        seq, tgt = make_recall_batch(64)
        loss = F.cross_entropy(m(seq), tgt)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        seq, tgt = make_recall_batch(2000)
        return (m(seq).argmax(-1) == tgt).float().mean().item()

widths = [16, 32, 64, 128, 256]
accs = [train_recall(d_ff) for d_ff in widths]
for w, a in zip(widths, accs):
    print(f'd_ff = {w:>4}  (d_ff/d_model = {w/D_MODEL_R:>5.2f}):  acc = {a:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([w/D_MODEL_R for w in widths], accs, 'o-')
ax.set_xlabel('d_ff / d_model'); ax.set_ylabel('accuracy')
ax.set_xscale('log'); ax.set_title('FFN width vs. recall')
plt.tight_layout(); plt.show()

**Sub-finding 3.** *At what width did the model click? Does the threshold relate to the task's required memory cells (5 pairs × something)?*

## Experiment 4 — activation ablation

In [ ]:
for act in ['gelu', 'relu', 'silu']:
    print(f'activation = {act:>5}: acc = {train_recall(d_ff=128, activation=act):.3f}')

**Sub-finding 4.** *Material difference, or noise-level? Confirms the literature view that activation choice is one of the smaller hyperparameter levers.*

## Finding

*3–6 sentences. Did attention-only fail XOR exactly as the linearity argument predicts? Did FFN width gate memorization on the recall task — and how does that threshold relate to the number of (key, value) pairs to memorize? Updated mental model: the FFN does **two** things — it's the only source of per-token nonlinearity (Experiment 2), and it's the model's key-value memory bank (Experiment 3). These are different roles in the same module.*